# AgentSmith: Multi-Bank Fraud Detection with LangGraph

Fraudsters move money in a circle across several banks so that no single bank sees the pattern. AgentSmith looks
at all three banks at once. Given one alert transaction it:

1. **Loads** the transaction and **detects** the payment chain it belongs to
2. **Enriches** every account in the chain with each bank's customer profile and history
3. **Scores** the risk with transparent rules (circular movement, amount decrement, new accounts, ...)
4. **Routes**: a score of 70 or more **pauses for a human analyst**, anything lower is auto-cleared
5. **Writes** a plain-English audit memo

In this notebook you will run the graph locally, handle the human-in-the-loop pause, publish a labelled dataset to
LangSmith, evaluate the agent, and call the version deployed on LangSmith.

## Step 0: Setup

Copy `.env.example` to `.env` in the repo root and fill in your LangSmith key and workspace. The agent itself is
rule-based, so it needs no model, but tracing, datasets and evals use LangSmith.

In [ ]:
# Notebook only: let Python find `agent.py` and the `utils/` and `tools/` folders.
# This notebook lives in 04_agentsmith/modules/, so the 04_agentsmith folder is one level up.
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from dotenv import load_dotenv

load_dotenv(project_root.parent / ".env")

## Step 1: Look at the data

Three banks each hold their own transactions. `fraud_labels.jsonl` is the ground truth, one label per payment chain.

In [ ]:
import json

from utils import db

print(db.table("banks"))
for bank in ["bank_a", "bank_b", "bank_c"]:
    print(bank, len(db.table(f"{bank}/transactions")), "transactions")

labels = db.table("fraud_labels")
for label in labels:
    print(label.get("chain_id") or label.get("transaction_id"), label["label"], label["pattern"])

## Step 2: The graph

```
START -> load_trigger -> detect_chain -> enrich_bank_a -> enrich_bank_b -> enrich_bank_c -> score_risk
                                                                                              |
                                                          score >= 70 -> human_review --+     |
                                                          score <  70 -> auto_clear   --+-> write_audit -> END
```

The nodes live in `utils/nodes.py`, the cross-bank lookups in `tools/network.py` and the per-bank enrichment in
`tools/bank.py`. A human pause needs a checkpointer when you run locally (LangGraph Platform adds one for you when
deployed), so we compile a local copy with an in-memory one.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

from agent import build_agent

local_agent = build_agent(checkpointer=InMemorySaver())
local_agent

## Step 3: A fraud alert pauses for a human

`TXN-A-001` is the first leg of `CHAIN-001`, a GBP 5,000 round trip A -> B -> C -> A in under five minutes.

In [ ]:
config = {"configurable": {"thread_id": "demo-chain-001"}}
result = local_agent.invoke({"alert_transaction_id": "TXN-A-001"}, config)

print("risk_score:", result["risk_score"])
for reason in result["risk_reasons"]:
    print(" -", reason)
print("paused for human:", "__interrupt__" in result)

The graph stopped at `human_review`. Resume it with the analyst's decision using `Command(resume=...)`.

In [ ]:
from langgraph.types import Command

final = local_agent.invoke(
    Command(resume={"decision": "investigate", "notes": "Classic round trip, escalate to MLRO."}), config
)
print(final["audit_memo"])

## Step 4: A legitimate alert is auto-cleared

`CHAIN-007` is a supplier invoice settlement between established accounts.

In [ ]:
result = local_agent.invoke(
    {"alert_transaction_id": "TXN-A-013"}, {"configurable": {"thread_id": "demo-chain-007"}}
)
print(result["audit_memo"])

## Step 5: Score every transaction

The scorer is deterministic, so we can run all 40 transactions in the three bank directories and compare against the
labels. This is what the LangSmith eval in the next steps automates.

In [ ]:
from collections import defaultdict

by_chain = {l["chain_id"]: l for l in labels if l.get("chain_id")}
by_txn = {l["transaction_id"]: l for l in labels if l.get("transaction_id")}

scores = defaultdict(set)
for bank_dir in ["bank_a", "bank_b", "bank_c"]:
    for txn in db.table(f"{bank_dir}/transactions"):
        label = by_txn.get(txn["transaction_id"]) or by_chain[txn["chain_id"]]
        out = local_agent.invoke(
            {"alert_transaction_id": txn["transaction_id"]},
            {"configurable": {"thread_id": f"all-{txn['transaction_id']}"}},
        )
        scores[(txn["chain_id"], label["label"], label["pattern"])].add(out["risk_score"])

for key, value in sorted(scores.items(), key=lambda kv: str(kv[0][0])):
    print(key, sorted(value))

**What this shows:** circular fraud (chains 001-003) scores 85-100 and is flagged. The two *layering* chains (004,
005) score 10 and 35 because no rule detects a sum being split and reassembled, so they are missed. The legitimate
payroll chain (006) scores 50, which is the closest a legitimate case gets to the threshold.

## Step 6: Publish the dataset to LangSmith

`evals/run_eval.py` builds one example per transaction (inputs: `alert_transaction_id`; reference outputs: chain,
label, pattern, expected decision) with a split per bank, and uploads it as the `agentsmith-fraud-alerts` dataset.
It is idempotent: if the dataset exists it is reused.

In [ ]:
from langsmith import Client

from evals.run_eval import DATASET, ensure_dataset

client = Client()
dataset = ensure_dataset(client)
print(dataset.name, dataset.id, "examples:", len(list(client.list_examples(dataset_id=dataset.id))))

## Step 7: Evaluate the agent

We score each run with code evaluators defined in `evals/run_eval.py`:

| Evaluator | Meaning |
|---|---|
| `flag_matches_label` | the agent flagged exactly the fraud cases |
| `false_negative` | fraud that was **not** flagged (the costly miss) |
| `false_positive` | legitimate activity that **was** flagged |
| `chain_detected` | the right payment chain was found |
| `reasons_for_flagged` | a flagged case always explains why |

Here we evaluate the *local* graph. A flagged run is left paused rather than resumed, so "flagged" means the agent
asked for a human.

In [ ]:
from langsmith import evaluate

from evals.run_eval import (
    chain_detected,
    false_negative,
    false_positive,
    flag_matches_label,
    reasons_for_flagged,
)


def local_target(inputs: dict) -> dict:
    thread = {"configurable": {"thread_id": f"eval-{inputs['alert_transaction_id']}"}}
    out = local_agent.invoke(inputs, thread)
    return {
        "chain_id": out.get("chain_id"),
        "risk_score": out["risk_score"],
        "risk_reasons": out["risk_reasons"],
        "flagged": "__interrupt__" in out,
    }


# Set to True to create a new experiment in LangSmith (about a minute).
RUN_LOCAL_EVAL = False

if RUN_LOCAL_EVAL:
    results = evaluate(
        local_target,
        data=DATASET,
        evaluators=[flag_matches_label, false_negative, false_positive, chain_detected, reasons_for_flagged],
        experiment_prefix="agentsmith-local",
        max_concurrency=4,
    )

Open the dataset in the LangSmith console (Datasets & Experiments -> `agentsmith-fraud-alerts` -> Experiments) to
compare experiments side by side.

## Step 8: Call the deployed agent

The same graph is deployed on LangSmith. Set `DEPLOYMENT_URL` to your deployment's URL (shown in the console under
Deployments). `run_eval.py` also accepts `--url` to run the whole evaluation against the deployment from the
command line. Deployment traces appear in the `agentsmith-production` project.

In [ ]:
import os

from langgraph_sdk import get_sync_client

DEPLOYMENT_URL = "https://agentsmith-1a092d08c74758e386290b904b63e00e.us.langgraph.app"

remote = get_sync_client(url=DEPLOYMENT_URL, api_key=os.environ["LANGSMITH_API_KEY"])
thread = remote.threads.create()
remote.runs.wait(thread["thread_id"], "agentsmith", input={"alert_transaction_id": "TXN-A-001"})

state = remote.threads.get_state(thread["thread_id"])
print("risk_score:", state["values"]["risk_score"], "| waiting on:", state["next"])

## Step 9: Online evaluators and next steps

- **Online evaluators** score live traces in `agentsmith-production` with no labels (for example "a flagged run
  must have reasons"). Add them in the console: project -> New -> Evaluator, filter to root runs.
- **Close the layering gap**: add a rule to `score_risk` that detects a fan-out (one account sending several
  transfers within minutes) that is reassembled at another account, redeploy with
  `langgraph deploy --name agentsmith`, then re-run the evaluation and watch `false_negative` fall.